# scPolASeq — Clustering & UMAP

Loads the STARsolo filtered matrices + cluster annotations produced by the pipeline  
and visualises cell clusters, QC metrics and marker genes.

Tested against run **pbmc1k_20260413034130** (KMeans 8-cluster mode).

In [ ]:
from pathlib import Path

# ── Configure paths ───────────────────────────────────────────────────────────
RESULTS_DIR = Path("/scratch/results/scPolASeq/pbmc1k")
LABELS_DIR  = RESULTS_DIR / "labels"
ALIGN_DIR   = RESULTS_DIR / "alignment"

# One entry per sequencing lane (library_id → Solo.out path)
LIBRARIES = {
    "pbmc_1k_v3_L001": ALIGN_DIR / "pbmc_1k_v3_L001.Solo.out",
    "pbmc_1k_v3_L002": ALIGN_DIR / "pbmc_1k_v3_L002.Solo.out",
}

# h5ad files (real anndata if pipeline ran with fixed script, JSON stub otherwise)
H5AD_FILES = [LABELS_DIR / f"{lid}.clustered.h5ad" for lid in LIBRARIES]

# Merged annotation table produced by BUILD_GROUP_MAP
CELL_ANNOTATIONS = LABELS_DIR / "cell_annotations.tsv"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(6, 5))
print("scanpy", sc.__version__)

## 1  Load data

Tries to load the real h5ad first. Falls back to reading the STARsolo filtered matrix  
and attaching cluster annotations from `cell_annotations.tsv`.

In [ ]:
import json

def _is_real_h5ad(path: Path) -> bool:
    """Return True only if the file is a proper HDF5/anndata, not a JSON stub."""
    if not path.exists():
        return False
    with open(path, "rb") as fh:
        magic = fh.read(8)
    return magic[:4] == b'\x89HDF' or magic[:4] == b'\x89\x48\x44\x46'  # HDF5 signature

def load_library(library_id: str, solo_out: Path, h5ad_path: Path) -> sc.AnnData:
    if _is_real_h5ad(h5ad_path):
        print(f"  {library_id}: loading real h5ad from {h5ad_path.name}")
        adata = sc.read_h5ad(str(h5ad_path))
        return adata

    # Fallback: read filtered matrix from STARsolo
    mtx = solo_out / "Gene" / "filtered"
    print(f"  {library_id}: loading matrix from {mtx}")
    adata = sc.read_10x_mtx(str(mtx), var_names="gene_symbols", cache=False)
    adata.obs["library_id"] = library_id
    return adata

print("Loading libraries…")
adatas = [load_library(lid, solo, h5ad)
          for (lid, solo), h5ad in zip(LIBRARIES.items(), H5AD_FILES)]
print("Done.")

In [ ]:
# Concatenate lanes
adata = sc.concat(
    adatas,
    label="library_id",
    keys=list(LIBRARIES.keys()),
    join="inner",
    merge="unique"
)
adata.obs_names_make_unique()
print(adata)

In [ ]:
# Attach cluster annotations from pipeline TSV
anno = pd.read_csv(CELL_ANNOTATIONS, sep="\t")
print("Annotation columns:", anno.columns.tolist())
print(anno.head(3))

# Build barcode → cluster_id lookup
bc_to_cluster  = anno.set_index("barcode_corrected")["cluster_id"].to_dict()
bc_to_celltype = anno.set_index("barcode_corrected")["cell_type"].to_dict()

# Strip the lane suffix that concat adds (format: barcode-1, barcode-2 ...)
def strip_suffix(obs_name):
    return obs_name.rsplit("-", 1)[0] if "-" in obs_name else obs_name

bare_bcs = [strip_suffix(n) for n in adata.obs_names]
adata.obs["cluster_id"] = [bc_to_cluster.get(bc, "unassigned") for bc in bare_bcs]
adata.obs["cell_type"]  = [bc_to_celltype.get(bc, "unlabeled")  for bc in bare_bcs]

print("\nCluster distribution:")
print(adata.obs["cluster_id"].value_counts().sort_index())

## 2  QC

In [ ]:
# Mitochondrial gene fraction
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(axes,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    ["Genes / cell",       "UMIs / cell",  "% MT reads"]):
    ax.hist(adata.obs[col], bins=60, edgecolor="none", color="steelblue", alpha=0.8)
    ax.set_xlabel(title)
    ax.set_ylabel("Cells")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 3  Preprocessing & embedding

If the h5ad already contains a UMAP (`X_umap`), it is used directly.  
Otherwise the standard scanpy pp → pca → neighbors → umap pipeline is run.

In [ ]:
if "X_umap" not in adata.obsm:
    print("Computing embedding…")
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
    adata_hvg = adata[:, adata.var["highly_variable"]].copy()
    adata_hvg.X = adata_hvg.X.toarray()
    sc.pp.scale(adata_hvg, max_value=10)
    n_comps = min(30, adata_hvg.n_obs - 1, adata_hvg.n_vars - 1)
    sc.tl.pca(adata_hvg, n_comps=n_comps)
    sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=n_comps)
    sc.tl.umap(adata_hvg)
    # Transfer embedding back to full adata
    obs_in_hvg = adata_hvg.obs_names
    idx = adata.obs_names.isin(obs_in_hvg)
    adata = adata[idx].copy()
    adata.obsm["X_pca"]  = adata_hvg.obsm["X_pca"]
    adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"]
    print(f"Embedding done: {adata.n_obs} cells retained after QC filters.")
else:
    print("Using pre-computed UMAP from h5ad.")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

print(adata)

## 4  UMAP visualisations

In [ ]:
sc.pl.umap(adata, color="cluster_id", title="KMeans clusters",
           palette="tab10", frameon=False, legend_loc="on data")

In [ ]:
sc.pl.umap(adata, color="library_id", title="Lane (batch)",
           palette="Set2", frameon=False)

In [ ]:
sc.pl.umap(adata, color=["n_genes_by_counts", "total_counts", "pct_counts_mt"],
           ncols=3, frameon=False, cmap="viridis")

## 5  Marker genes per cluster

In [ ]:
# Wilcoxon rank-sum test — one-vs-rest per cluster
sc.tl.rank_genes_groups(adata, groupby="cluster_id", method="wilcoxon",
                        use_raw=False, n_genes=20)
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
# Top markers as a tidy DataFrame
marker_df = sc.get.rank_genes_groups_df(adata, group=None)
top_markers = (
    marker_df[marker_df["pvals_adj"] < 0.05]
    .sort_values(["group", "scores"], ascending=[True, False])
    .groupby("group").head(5)
)
top_markers

In [ ]:
# Dot plot: top 3 markers per cluster
marker_genes = (
    top_markers.groupby("group").head(3)["names"]
    .drop_duplicates().tolist()
)
if marker_genes:
    sc.pl.dotplot(adata, marker_genes, groupby="cluster_id",
                  standard_scale="var", dendrogram=False)

## 6  Known PBMC marker panel

Quick visual sanity-check against canonical human PBMC markers.

In [ ]:
PBMC_MARKERS = {
    "T CD4"   : ["CD3D", "CD3E", "CD4"],
    "T CD8"   : ["CD3D", "CD8A", "CD8B"],
    "NK"      : ["GNLY", "NKG7", "NCAM1"],
    "B cell"  : ["MS4A1", "CD79A", "CD19"],
    "Monocyte": ["CD14", "LYZ", "S100A8"],
    "DC"      : ["FCER1A", "CST3"],
    "Platelet": ["PPBP", "PF4"],
}
flat_markers = [g for genes in PBMC_MARKERS.values() for g in genes]
present = [g for g in flat_markers if g in adata.var_names]
print(f"{len(present)}/{len(flat_markers)} canonical markers found in dataset")

if present:
    sc.pl.dotplot(adata, present, groupby="cluster_id",
                  standard_scale="var", dendrogram=True,
                  title="Canonical PBMC markers")

In [ ]:
# Feature (gene expression) UMAP for key markers
highlight = [g for g in ["CD3D", "MS4A1", "CD14", "NKG7", "CD8A"] if g in adata.var_names]
if highlight:
    sc.pl.umap(adata, color=highlight, ncols=3, frameon=False, cmap="Reds")

## 7  Cluster summary table

In [ ]:
summary = (
    adata.obs
    .groupby("cluster_id")
    .agg(
        n_cells=("cluster_id", "count"),
        median_genes=("n_genes_by_counts", "median"),
        median_umi=("total_counts", "median"),
        median_pct_mt=("pct_counts_mt", "median"),
    )
    .sort_index()
)
summary["pct_cells"] = (summary["n_cells"] / summary["n_cells"].sum() * 100).round(1)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
summary["n_cells"].plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Cluster")
ax.set_ylabel("# cells")
ax.set_title("Cells per KMeans cluster")
plt.tight_layout()
plt.show()